# NumPy for Machine Learning — N01–N06

Trusted canonical notebook, content python.numpy.1.0.0. Predict before running, inspect coordinate meaning, then compare the assertions. Execution/download never awards platform mastery. No credentials, uploads, external datasets or hosted learner execution. Genuine Google Colab execution remains UNVERIFIED.

Use the approved exact local profile. Restart the kernel and Run All; no hidden variables or unfinished TODOs are required. Suggested private practice: change one declared value after predicting its effect, then restart to restore the canonical outputs.


In [ ]:
import sys, json, warnings, hashlib, platform, time, resource
from importlib.metadata import version
import numpy as np
assert sys.version_info[:3] == (3, 13, 16)
profile = {name: version(name) for name in ['numpy', 'pandas', 'matplotlib', 'scikit-learn']}
assert profile == {'numpy': '2.5.4', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'scikit-learn': '1.9.1'}
rng = np.random.default_rng(42)
started = time.monotonic()
records = [{'id': 'H1', 'area': 2, 'age': 10}, {'id': 'H2', 'area': 1, 'age': 5}, {'id': 'H3', 'area': 3, 'age': 15}]
manifest = {'id': 'numpy-houses-teaching.v1', 'origin': 'original fictional teaching records; no personal data', 'permission': 'authored for platform teaching', 'row_ids': ['H1', 'H2', 'H3'], 'columns': ['area', 'age'], 'units': ['area units', 'years'], 'missing_policy': 'explicit named derived fixtures only', 'limitations': 'tiny teaching data, no performance validation'}
manifest['sha256'] = hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()
all_results = []


In [ ]:
def normalize(value):
    if isinstance(value, np.ndarray):
        assert value.ndim <= 2 and value.size <= 64 and all(size <= 32 for size in value.shape)
        assert str(value.dtype) in ("int64", "float64", "bool")
        return {"shape": list(value.shape), "dtype": str(value.dtype), "values": normalize(value.tolist())}
    if isinstance(value, np.generic):
        return normalize(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return {"tag": "missing"} if np.isnan(value) else {"tag": "nonfinite", "value": "+inf" if value > 0 else "-inf"}
    if isinstance(value, dict):
        return {key: normalize(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize(item) for item in value]
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        assert abs(value) <= 1e12
    return value



## N01 — Arrays describe data and shape

Shape records what each axis represents, while dtype constrains stored values.

Mathematical connection: Rows are observations; columns are ordered features.

Recall: P01–P02, P05; Linear Algebra vectors/matrices.

Misconception to investigate: Number of rows is not ndim; reshaping does not create information or preserve semantic row meaning.


### N01.O1 — Distinguish list behavior from numeric array behavior.

Predict the values, axes, dtype and any named error before execution. A Python list repeats its elements; a numeric ndarray multiplies each stored value. Equal-looking containers do not promise the same arithmetic.

Changed-case practice: Compare type-tagged traces.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    values = [3, 4]
    array = np.array(values, dtype=np.int64)
    result = [values * 2, array * 2]
observed = normalize(result)
expected = json.loads("[[3,4,3,4],{\"shape\":[2],\"dtype\":\"int64\",\"values\":[6,8]}]")
assert observed == expected
all_results.append({'objective': 'N01.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N01.O2 — Read rows/features and predict shape, ndim and size.

Predict the values, axes, dtype and any named error before execution. There are three observations and two ordered features: shape (3,2), ndim 2 and size 6. Axis 0 locates a row, axis 1 a feature; ndim is not the number of observations.

Changed-case practice: Label a small matrix with row IDs and feature names.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    result = {'shape': X.shape, 'ndim': X.ndim, 'size': X.size, 'axis0': 'observations', 'axis1': 'features'}
observed = normalize(result)
expected = json.loads("{\"shape\":[3,2],\"ndim\":2,\"size\":6,\"axis0\":\"observations\",\"axis1\":\"features\"}")
assert observed == expected
all_results.append({'objective': 'N01.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N01.O3 — Check element-count compatibility before reshape.

Predict the values, axes, dtype and any named error before execution. Reshape preserves six elements in order, not their observation grouping. (2,3) fits; (2,4) needs eight elements and raises ValueError.

Changed-case practice: Count elements before drawing the new grid.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    valid = X.reshape(2, 3)
    try:
        invalid = X.reshape(2, 4)
    except ValueError:
        error = {'tag': 'error', 'class': 'ValueError'}
    result = [valid, error, 'row meaning changed']
observed = normalize(result)
expected = json.loads("[{\"shape\":[2,3],\"dtype\":\"int64\",\"values\":[[3,10,4],[5,5,15]]},{\"tag\":\"error\",\"class\":\"ValueError\"},\"row meaning changed\"]")
assert observed == expected
all_results.append({'objective': 'N01.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N01.O4 — Choose numeric dtype and recognize truncating integer conversion.

Predict the values, axes, dtype and any named error before execution. The requested int64 conversion truncates toward zero, including negative values. It loses decimal measurements; it is not ordinary rounding or a harmless display change.

Changed-case practice: Inspect original values against the dtype result.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    values = np.array([3.75, -3.75], dtype=np.float64)
    result = [values, values.astype(np.int64)]
observed = normalize(result)
expected = json.loads("[{\"shape\":[2],\"dtype\":\"float64\",\"values\":[3.75,-3.75]},{\"shape\":[2],\"dtype\":\"int64\",\"values\":[3,-3]}]")
assert observed == expected
all_results.append({'objective': 'N01.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## N02 — Indexing, masks, views and copies

Selection changes axes and may share data with its source.

Mathematical connection: Selecting a row versus a column changes vector shape and meaning.

Recall: N01, P02–P03.

Misconception to investigate: (3,) and (3,1) are not interchangeable; a selected array is not always an independent copy.


### N02.O1 — Predict values and shape after row/column indexing.

Predict the values, axes, dtype and any named error before execution. An integer index removes the selected axis; a length-one slice retains it. Inspect the coordinates and exact result shapes, not only the displayed numbers.

Changed-case practice: Draw retained axes and their lengths.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    result = [X[:, 0], X[:, 0:1]]
observed = normalize(result)
expected = json.loads("[{\"shape\":[3],\"dtype\":\"int64\",\"values\":[3,4,5]},{\"shape\":[3,1],\"dtype\":\"int64\",\"values\":[[3],[4],[5]]}]")
assert observed == expected
all_results.append({'objective': 'N02.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N02.O2 — Apply a Boolean mask to matching observations.

Predict the values, axes, dtype and any named error before execution. Apply the same observation mask to X, y and row IDs. The retained targets belong to H2 and H3, not merely the first two original targets.

Changed-case practice: Compare mask length and selected row IDs.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    y = np.array([100, 200, 300], dtype=np.int64)
    row_ids = ['H1', 'H2', 'H3']
    mask = X[:, 0] > 3
    result = {'rows': [row_id for row_id, keep in zip(row_ids, mask) if keep], 'X': X[mask], 'y': y[mask]}
observed = normalize(result)
expected = json.loads("{\"rows\":[\"H2\",\"H3\"],\"X\":{\"shape\":[2,2],\"dtype\":\"int64\",\"values\":[[4,5],[5,15]]},\"y\":{\"shape\":[2],\"dtype\":\"int64\",\"values\":[200,300]}}")
assert observed == expected
all_results.append({'objective': 'N02.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N02.O3 — Distinguish shared view edits from explicit copy edits.

Predict the values, axes, dtype and any named error before execution. This basic slice shares source storage; editing its first value changes X. The explicit copy keeps the values from before the edit. Selection is not automatically an independent copy.

Changed-case practice: Trace source and view storage relationships.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    view = X[:, 0]
    copy = X[:, 0].copy()
    view[0] = 11
    result = [X, view, copy]
observed = normalize(result)
expected = json.loads("[{\"shape\":[3,2],\"dtype\":\"int64\",\"values\":[[11,10],[4,5],[5,15]]},{\"shape\":[3],\"dtype\":\"int64\",\"values\":[11,4,5]},{\"shape\":[3],\"dtype\":\"int64\",\"values\":[3,4,5]}]")
assert observed == expected
all_results.append({'objective': 'N02.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N02.O4 — Explain mask conjunction with parentheses and array operators.

Predict the values, axes, dtype and any named error before execution. Parenthesize each comparison and combine Boolean arrays with elementwise &. Python and asks for one truth value for the whole array and raises ValueError here.

Changed-case practice: Contrast scalar and with elementwise &.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    mask = (X[:, 0] > 3) & (X[:, 1] < 20)
    try:
        broken = (X[:, 0] > 3) and (X[:, 1] < 20)
    except ValueError:
        error = {'tag': 'error', 'class': 'ValueError'}
    result = [mask, error]
observed = normalize(result)
expected = json.loads("[{\"shape\":[3],\"dtype\":\"bool\",\"values\":[false,true,true]},{\"tag\":\"error\",\"class\":\"ValueError\"}]")
assert observed == expected
all_results.append({'objective': 'N02.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## N03 — Broadcasting is shape reasoning

Broadcasting aligns coordinates; legal arithmetic can still describe the wrong operation.

Mathematical connection: Bias addition; rowwise versus columnwise offsets.

Recall: N01–N02.

Misconception to investigate: Matching total element counts does not establish broadcast compatibility; a legal broadcast may still represent the wrong operation.


### N03.O1 — Align trailing dimensions and apply singleton expansion.

Predict the values, axes, dtype and any named error before execution. Align trailing dimensions: (3,2)+(2,) supplies one offset per feature. (3,) does not match trailing length 2; total element count does not establish compatibility.

Changed-case practice: Right-align dimensions and mark equality or 1.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    bias = np.array([1, 3], dtype=np.int64)
    valid = X + bias
    try:
        invalid = X + np.array([1, 2, 3], dtype=np.int64)
    except ValueError:
        error = {'tag': 'error', 'class': 'ValueError'}
    result = [valid, error]
observed = normalize(result)
expected = json.loads("[{\"shape\":[3,2],\"dtype\":\"int64\",\"values\":[[4,13],[5,8],[6,18]]},{\"tag\":\"error\",\"class\":\"ValueError\"}]")
assert observed == expected
all_results.append({'objective': 'N03.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N03.O2 — Select the intended bias orientation rather than merely legal arithmetic.

Predict the values, axes, dtype and any named error before execution. The square table makes both expressions legal. Per-row offsets need shape (2,1), so each observation receives its own offset across both features; shape legality alone cannot establish intent.

Changed-case practice: Attach a named meaning to each offset dimension.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5]], dtype=np.int64)
    offsets = np.array([1, 2], dtype=np.int64)
    result = {'intended_per_row': X + offsets.reshape(2, 1), 'legal_but_per_feature': X + offsets}
observed = normalize(result)
expected = json.loads("{\"intended_per_row\":{\"shape\":[2,2],\"dtype\":\"int64\",\"values\":[[4,11],[6,7]]},\"legal_but_per_feature\":{\"shape\":[2,2],\"dtype\":\"int64\",\"values\":[[4,12],[5,7]]}}")
assert observed == expected
all_results.append({'objective': 'N03.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N03.O3 — Predict an outer-style result from (m,1)+(n,).

Predict the values, axes, dtype and any named error before execution. (2,1)+(3,) expands the singleton coordinate into a (2,3) grid of pairwise sums. It is not a length-two paired addition.

Changed-case practice: Expand the small grid of pairwise sums.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    column = np.array([[3], [4]], dtype=np.int64)
    row = np.array([10, 20, 30], dtype=np.int64)
    result = column + row
observed = normalize(result)
expected = json.loads("{\"shape\":[2,3],\"dtype\":\"int64\",\"values\":[[13,23,33],[14,24,34]]}")
assert observed == expected
all_results.append({'objective': 'N03.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N03.O4 — Recognize an incompatible operation and choose a deliberate repair.

Predict the values, axes, dtype and any named error before execution. Reshape to (3,1) deliberately labels one offset per observation, then broadcast across features. A reshape repairs intent only when the coordinate meaning is justified.

Changed-case practice: Check both shape legality and data interpretation.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10], [4, 5], [5, 15]], dtype=np.int64)
    offsets = np.array([1, 2, 6], dtype=np.int64)
    result = [X + offsets.reshape(3, 1), 'deliberate per-row reshape']
observed = normalize(result)
expected = json.loads("[{\"shape\":[3,2],\"dtype\":\"int64\",\"values\":[[4,11],[6,7],[11,21]]},\"deliberate per-row reshape\"]")
assert observed == expected
all_results.append({'objective': 'N03.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## N04 — Vectorized predictions replace repeated scalar work

Vectorization packages the same ordered weighted predictions for many observations.

Mathematical connection: Xw+b and residual ŷ−y; distinguish multiplication from reduction.

Recall: N02–N03; dot products.

Misconception to investigate: Vectorization does not mean every * is a dot product; broadcasting bias in the wrong place can double-count it.


### N04.O1 — Map a scalar loop to elementwise contributions.

Predict the values, axes, dtype and any named error before execution. Each cell is a feature's weighted contribution in $k. Elementwise multiplication still has one column per feature; no sum or bias has yet produced a prediction.

Changed-case practice: Review dot-product term multiplication before summing.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[2, 10], [1, 5]], dtype=np.float64)
    w = np.array([100, -2], dtype=np.float64)
    b = 50
    result = X * w
observed = normalize(result)
expected = json.loads("{\"shape\":[2,2],\"dtype\":\"float64\",\"values\":[[200,-20],[100,-10]]}")
assert observed == expected
all_results.append({'objective': 'N04.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N04.O2 — Compute ordered vectorized predictions with bias once per observation.

Predict the values, axes, dtype and any named error before execution. Contract the feature axis with w, then add b once per observation. Adding b to every contribution before summing counts it twice here. Feature order must match the coefficient order.

Changed-case practice: Compare each vector result with the scalar row formula.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[2, 10], [1, 5]], dtype=np.float64)
    w = np.array([100, -2], dtype=np.float64)
    b = 50
    prediction = X @ w + b
    repeated_bias = (X * w + b).sum(axis=1)
    result = {'prediction': prediction, 'repeated_bias': repeated_bias}
observed = normalize(result)
expected = json.loads("{\"prediction\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[230,140]},\"repeated_bias\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[280,190]}}")
assert observed == expected
all_results.append({'objective': 'N04.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N04.O3 — Keep residual sign and feature/target row pairing.

Predict the values, axes, dtype and any named error before execution. Residual is prediction minus the target for the same row, matching the existing Regression convention. Reordering targets or reversing the sign changes the meaning even when lengths match.

Changed-case practice: Match row IDs and the existing Regression residual convention.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[2, 10], [1, 5]], dtype=np.float64)
    w = np.array([100, -2], dtype=np.float64)
    b = 50
    row_ids = ['H1', 'H2']
    y = np.array([200, 150], dtype=np.float64)
    result = {'rows': row_ids, 'prediction': X @ w + b, 'residual': X @ w + b - y}
observed = normalize(result)
expected = json.loads("{\"rows\":[\"H1\",\"H2\"],\"prediction\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[230,140]},\"residual\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[30,-10]}}")
assert observed == expected
all_results.append({'objective': 'N04.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N04.O4 — Explain equivalence rather than assert universal speed superiority.

Predict the values, axes, dtype and any named error before execution. Compare ordered values and shapes against the scalar row formula. Equivalence is a numerical claim; a tiny example does not establish universal performance superiority.

Changed-case practice: Check values/shapes first; do not grade timing.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[2, 10], [1, 5]], dtype=np.float64)
    w = np.array([100, -2], dtype=np.float64)
    b = 50
    loop = []
    for row in X:
        total = 0
        for value, weight in zip(row, w):
            total += value * weight
        loop.append(total + b)
    loop = np.array(loop, dtype=np.float64)
    vector = X @ w + b
    result = [loop, vector, loop.shape == vector.shape and bool(np.allclose(loop, vector))]
observed = normalize(result)
expected = json.loads("[{\"shape\":[2],\"dtype\":\"float64\",\"values\":[230,140]},{\"shape\":[2],\"dtype\":\"float64\",\"values\":[230,140]},true]")
assert observed == expected
all_results.append({'objective': 'N04.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## N05 — Matrix multiplication and transposes

Matrix multiplication contracts one shared axis rather than multiplying cells independently.

Mathematical connection: (m,n)@(n,)→(m,); (m,n)@(n,k)→(m,k).

Recall: N01–N04; matrix compatibility.

Misconception to investigate: @ is not *; transpose does not arbitrarily fix feature meaning; multiplication order matters.


### N05.O1 — Check legal dimensions for matrix–vector and matrix–matrix products.

Predict the values, axes, dtype and any named error before execution. For (m,n)@(n,k), the shared feature length n must match and output shape is (m,k). A vector (n,) gives (m,); matching outer lengths cannot repair an inner mismatch.

Changed-case practice: Pair the inner dimensions and keep outer dimensions.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    A = np.array([[1, 2, 3], [3, 4, 5]], dtype=np.int64)
    B = np.array([[1, 0], [0, 1], [1, 1]], dtype=np.int64)
    v = np.array([1, 2, 3], dtype=np.int64)
    try:
        invalid = A @ np.array([[1, 0], [0, 1]], dtype=np.int64)
    except ValueError:
        error = {'tag': 'error', 'class': 'ValueError'}
    result = [A @ B, A @ v, error]
observed = normalize(result)
expected = json.loads("[{\"shape\":[2,2],\"dtype\":\"int64\",\"values\":[[4,5],[8,9]]},{\"shape\":[2],\"dtype\":\"int64\",\"values\":[14,26]},{\"tag\":\"error\",\"class\":\"ValueError\"}]")
assert observed == expected
all_results.append({'objective': 'N05.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N05.O2 — Distinguish elementwise multiplication from matrix multiplication.

Predict the values, axes, dtype and any named error before execution. @ and np.matmul contract the matching feature axis into one dot product per row. * preserves a contribution for every cell; it does not perform the reduction.

Changed-case practice: Expand one row's scalar dot product.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    A = np.array([[1, 2], [3, 4]], dtype=np.int64)
    w = np.array([10, 1], dtype=np.int64)
    result = {'matrix': A @ w, 'elementwise': A * w, 'equivalent': np.matmul(A, w)}
observed = normalize(result)
expected = json.loads("{\"matrix\":{\"shape\":[2],\"dtype\":\"int64\",\"values\":[12,34]},\"elementwise\":{\"shape\":[2,2],\"dtype\":\"int64\",\"values\":[[10,2],[30,4]]},\"equivalent\":{\"shape\":[2],\"dtype\":\"int64\",\"values\":[12,34]}}")
assert observed == expected
all_results.append({'objective': 'N05.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N05.O3 — Predict a transpose and its feature/observation interpretation.

Predict the values, axes, dtype and any named error before execution. Transpose swaps axis coordinates and their meanings: the feature axis comes first. It does not change which weight or target belongs to a house.

Changed-case practice: Swap axis labels rather than just copying numbers.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[3, 10, 20], [4, 5, 15]], dtype=np.int64)
    result = {'transposed': X.T, 'axis0': 'features', 'axis1': 'observations'}
observed = normalize(result)
expected = json.loads("{\"transposed\":{\"shape\":[3,2],\"dtype\":\"int64\",\"values\":[[3,4],[10,5],[20,15]]},\"axis0\":\"features\",\"axis1\":\"observations\"}")
assert observed == expected
all_results.append({'objective': 'N05.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N05.O4 — Recognize order-dependent products and reject a meaningless inverse shortcut.

Predict the values, axes, dtype and any named error before execution. For column vectors apply A then B using B@A@v; order changes the result. An inverse is not a general repair for shape or meaning, nor the default numerical fitting method.

Changed-case practice: Recall transformation composition and numerical solvers.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    A = np.array([[2, 0], [0, 1]], dtype=np.int64)
    B = np.array([[1, 3], [0, 1]], dtype=np.int64)
    v = np.array([1, 2], dtype=np.int64)
    result = {'A_then_B': B @ A @ v, 'B_then_A': A @ B @ v}
observed = normalize(result)
expected = json.loads("{\"A_then_B\":{\"shape\":[2],\"dtype\":\"int64\",\"values\":[8,2]},\"B_then_A\":{\"shape\":[2],\"dtype\":\"int64\",\"values\":[14,2]}}")
assert observed == expected
all_results.append({'objective': 'N05.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## N06 — Reductions, missingness and numeric limits

Reduction removes an axis and changes the denominator/units of the summary.

Mathematical connection: Sample versus population variance; reduction outputs and units.

Recall: N02–N05; Statistics center/spread.

Misconception to investigate: Axis 0 reduces over observations; NaN is not zero; default variance conventions differ across libraries.


### N06.O1 — Predict reduction values and result shape.

Predict the values, axes, dtype and any named error before execution. axis=0 reduces across observations, leaving a summary per feature. axis=1 reduces across features, leaving a value per row. keepdims retains a singleton axis for deliberate broadcasting; unlike-feature row means may lack a useful shared unit.

Changed-case practice: Cross out the reduced axis.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[1, 10], [3, 20]], dtype=np.float64)
    result = {'feature_mean': X.mean(axis=0), 'row_mean': X.mean(axis=1), 'kept_axis': X.mean(axis=0, keepdims=True), 'feature_sum': X.sum(axis=0)}
observed = normalize(result)
expected = json.loads("{\"feature_mean\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[2,15]},\"row_mean\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[5.5,11.5]},\"kept_axis\":{\"shape\":[1,2],\"dtype\":\"float64\",\"values\":[[2,15]]},\"feature_sum\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[4,30]}}")
assert observed == expected
all_results.append({'objective': 'N06.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N06.O2 — Choose the correct axis and denominator for a stated statistic.

Predict the values, axes, dtype and any named error before execution. There are two observations: population variance divides by 2, sample variance by 2-1. The mean retains the original unit; variance has squared units. State ddof, denominator and intended population before interpreting a value.

Changed-case practice: Count contributing observations and revisit sample/population definitions.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    values = np.array([1, 3], dtype=np.float64)
    result = {'mean': values.mean(), 'population': values.var(ddof=0), 'sample': values.var(ddof=1), 'n': values.size}
observed = normalize(result)
expected = json.loads("{\"mean\":2,\"population\":1,\"sample\":2,\"n\":2}")
assert observed == expected
all_results.append({'objective': 'N06.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N06.O3 — Distinguish missing, zero and nonfinite outputs.

Predict the values, axes, dtype and any named error before execution. NaN is missing, not zero. nanmean excludes missing cells from its denominator, but an all-missing feature still has no observed mean; the named missing outcome is not an invented zero.

Changed-case practice: Compare value masks rather than formatted display.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    X = np.array([[0, np.nan], [1, np.nan]], dtype=np.float64)
    result = {'missing': np.isnan(X), 'ordinary': X.mean(axis=0), 'nan_aware': np.nanmean(X, axis=0)}
observed = normalize(result)
expected = json.loads("{\"missing\":{\"shape\":[2,2],\"dtype\":\"bool\",\"values\":[[false,true],[false,true]]},\"ordinary\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[0.5,{\"tag\":\"missing\"}]},\"nan_aware\":{\"shape\":[2],\"dtype\":\"float64\",\"values\":[0.5,{\"tag\":\"missing\"}]}}")
assert observed == expected
all_results.append({'objective': 'N06.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### N06.O4 — Identify unsafe dtype/finite assumptions without changing old math semantics.

Predict the values, axes, dtype and any named error before execution. Finite inputs do not guarantee a finite result: division by zero produces a tagged nonfinite outcome here, with an explicit warning. Reject it as a usable model statistic; do not replace it with zero. Inspect dtype and finite checks before trusting a calculation.

Changed-case practice: Inspect dtype bounds and explicit finite checks.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import numpy as np
    values = np.array([1.0, 0.0], dtype=np.float64)
    ratio = values[0] / values[1]
    result = {'value': ratio, 'finite': bool(np.isfinite(ratio)), 'valid_model_statistic': bool(np.isfinite(ratio))}
observed = normalize(result)
expected = json.loads("{\"value\":{\"tag\":\"nonfinite\",\"value\":\"+inf\"},\"finite\":false,\"valid_model_statistic\":false}")
assert observed == expected
all_results.append({'objective': 'N06.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## Reproducibility and anchors

The canonical cases above assert all 24 objective outcomes. These independent analytic anchors connect the notebook to the approved Mathematics semantics. A local pass does not establish unrestricted coding fluency or genuine Colab compatibility.


In [ ]:
X = np.array([[2, 10], [1, 5]], dtype=np.float64)
w = np.array([100, -2], dtype=np.float64)
assert np.allclose(X @ w + 50, [230, 140])
assert np.array_equal(np.array([[1, 2], [3, 4]], dtype=np.int64) @ np.array([10, 1], dtype=np.int64), [12, 34])
assert np.allclose(np.array([[1, 10], [3, 20]], dtype=np.float64).mean(axis=0), [2, 15])
assert np.var([1, 3], ddof=0) == 1 and np.var([1, 3], ddof=1) == 2
assert len(all_results) == 24 and len({item['objective'] for item in all_results}) == 24
assert time.monotonic() - started < 300
summary = {'families': 24, 'lessons': 6, 'seed': 42, 'profile': profile, 'manifest': manifest, 'anchors': {'prediction': [230, 140], 'product': [12, 34], 'means': [2, 15], 'variance': [1, 2]}, 'examples': all_results}
print(json.dumps(summary, sort_keys=True, allow_nan=False))
